Note: Below code is work-in-progress (Fraukje, 14-01-2026)

## 1. Setup & Data Loading {#setup--data-loading}

Import required libraries and load the feature-selected dataset and baseline models from previous steps.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import joblib

# Import machine learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.neural_network import MLPClassifier
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns

# Import Bayesian optimization libraries
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

### Load Processed Data

In [ ]:
# Define data directory
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")

# Load data and models
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Load 'model_comparison_analysis.pkl'
model_data = joblib.load(PROCESSED_DIR / 'model_comparison_analysis.pkl')

## 2. Model Selection {#model-selection}

Based on the model comparison results, select the top performing models for hyperparameter tuning.

In [ ]:
# Extract models to tune from the loaded data
cv_results = model_data['cv_results']
models_to_tune = model_data['final_models']

print("Models selected for hyperparameter tuning:")
for name in models_to_tune.keys():
    print(f"- {name}")

In [ ]:
# Exclude decision tree due to poor performance during comparison
if 'DecisionTree' in models_to_tune:
    del models_to_tune['DecisionTree']

## 3. Hyperparameter Search Configuration {#hyperparameter-search-configuration}

Define hyperparameter search spaces for each selected model. \
Note: specific search spaces need to be determined. 
- many options available for loglinreg (e.g. for the solver), but not all combinations are compatible. 

In [ ]:
# Random forest search space definition
rf_search_space = {
    'n_estimators': Integer(50, 500, name='n_estimators'),
    'max_depth': Integer(5, 50, name='max_depth'),
    'min_samples_split': Integer(2, 20, name='min_samples_split'),
    'min_samples_leaf': Integer(1, 10, name='min_samples_leaf'),
    'max_features': Categorical(['sqrt', 'log2', None], name='max_features'),
    'bootstrap': Categorical([True, False], name='bootstrap'),
    'class_weight': Categorical(['balanced', None], name='class_weight')
}
print("Random Forest hyperparameter search space defined.")

# XGBoost search space definition
xgb_search_space = {
    'n_estimators': Integer(50, 500, name='n_estimators'),
    'max_depth': Integer(3, 20, name='max_depth'),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform', name='learning_rate'),
    'subsample': Real(0.5, 1.0, prior='uniform', name='subsample'),
    'colsample_bytree': Real(0.5, 1.0, prior='uniform', name='colsample_bytree'),
    'gamma': Real(0, 5, prior='uniform', name='gamma'),
    'reg_alpha': Real(0, 1, prior='uniform', name='reg_alpha'),
    'reg_lambda': Real(0, 1, prior='uniform', name='reg_lambda'),
    'scale_pos_weight': Real(1, 10, prior='uniform', name='scale_pos_weight')
}
print("XGBoost hyperparameter search space defined.")

# Logistic regression search space definition
lr_search_space = {
    'C': Real(0.01, 100, prior='log-uniform', name='C'),
    'penalty': Categorical(['l1', 'l2'], name='penalty'),
    'solver': Categorical(['saga', 'liblinear'], name='solver'),
    'max_iter': Integer(100, 1000, name='max_iter')
}
print("Logistic Regression hyperparameter search space defined.")

# Neural network search space definition
nn_search_space = {
    'hidden_layer_sizes': Categorical([(50,), (100,), (50, 50), (100, 50), (100, 100)], name='hidden_layer_sizes'),
    'activation': Categorical(['relu', 'tanh', 'logistic'], name='activation'),
    'solver': Categorical(['adam', 'sgd', 'lbfgs'], name='solver'),
    'alpha': Real(0.0001, 0.1, prior='log-uniform', name='alpha'),
    'learning_rate_init': Real(0.0001, 0.1, prior='log-uniform', name='learning_rate_init'),
    'max_iter': Integer(200, 1000, name='max_iter')
}
print("Neural Network hyperparameter search space defined.")

## 4. Hyperparameter Optimization {#hyperparameter-optimization}

Perform hyperparameter optimization using different search strategies.

### 4.1 Define strategy, scoring & set-up data

In [ ]:
# Set up cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scoring metric
scoring = 'f1'

print(f"Cross-validation strategy: {cv_strategy}")
print(f"Scoring metric: {scoring}")

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

# Define target and feature columns
exclude_cols = ['set', target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Target column: {target_col}")
print(f"Number of features: {len(feature_cols)}")
print(f"Excluded columns: {exclude_cols}")

# Prepare feature matrices and target vectors
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# Combine train and validation for cross-validation
X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

print(f"\nCombined train+val shape for cross-validation: {X_train_val.shape}")
print(f"Test set shape: {X_test.shape}")

### 4.2 Bayesian Optimization (Optional) {#bayesian-optimization-optional}

Advanced optimization using Bayesian methods for potentially more efficient search. \
Using Scikit-Optimize.

#### 4.2.1 Logistic regression

In [ ]:
# Bayesian Optimization for logistic regression
lr_bayes = BayesSearchCV(
    estimator=models_to_tune['Logistic Regression'],
    search_spaces=lr_search_space,
    n_iter=50,  # Number of parameter settings to sample
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
print("Starting Bayesian optimization for Logistic Regression...")

lr_bayes.fit(X_train_val, y_train_val)
print("Bayesian optimization for Logistic Regression completed.")
print(f"\nBest LR parameters (Bayesian): {lr_bayes.best_params_}")
print(f"Best LR score (Bayesian): {lr_bayes.best_score_:.4f}")

# Store results
bayesian_search_results['Logistic Regression'] = lr_bayes

# Print best score
print(f"Bayesian Search score: {lr_bayes.best_score_:.4f}")

In [ ]:
# Plot convergence after fitting
from skopt.plots import plot_convergence
plot_convergence(lr_bayes.optimizer_results_[0])

In [ ]:
# Apply best model to test set
best_lr_model = lr_bayes.best_estimator_
y_test_pred = best_lr_model.predict(X_test)
y_test_proba = best_lr_model.predict_proba(X_test)[:, 1]

# Calculate and print test set metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_proba)
print("\nTest Set Performance (Logistic Regression with Bayesian Optimization):")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print(f"ROC AUC: {test_roc_auc:.4f}")

#### 4.2.2 Random forest

In [ ]:
# Bayesian Optimization for Random Forest
rf_bayes = BayesSearchCV(
    estimator=models_to_tune['Random Forest'],
    search_spaces=rf_search_space,
    n_iter=50,  # Number of parameter settings to sample
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Perform Bayesian optimization
print("Performing Bayesian Optimization for Random Forest...")
rf_bayes.fit(X_train, y_train)

print(f"\nBest RF parameters (Bayesian): {rf_bayes.best_params_}")
print(f"Best RF score (Bayesian): {rf_bayes.best_score_:.4f}")

# Store results
bayesian_search_results = {'RandomForest': rf_bayes}

# Print best score
print(f"Bayesian Search score: {rf_bayes.best_score_:.4f}")

In [ ]:
# Plot convergence after fitting
from skopt.plots import plot_convergence
plot_convergence(rf_bayes.optimizer_results_[0])

## 5. Model Evaluation {#model-evaluation}

Evaluate and compare the performance of tuned models.